# Orquestador Silver a Gold

Gold organiza el Silver validado en hechos, dimensiones y metricas mensuales para Finanzas y Ventas.
Ejecutar despues de finalizar Silver, con un kernel nuevo. El YAML controla tipos, rutas y tablas exportadas.
Cada ejecucion publica una version inmutable; current.json cambia solo al completar la publicacion.
El CSV compacto se entrega en exports/gold del host. Recrear los contenedores con podman-compose up -d para aplicar el volumen exports si aun no esta montado.
Antes de publicar otra version, terminar la actualizacion de Power BI. Consultar docs/power_bi_gold.md para relaciones y medidas.

In [1]:
from pathlib import Path
import sys

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "jobs" / "gold_processor.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Ejecuta desde el proyecto o notebooks/.")
sys.path.insert(0, str(root / "jobs"))
from gold_processor import GoldProcessor
config_path = root / "jobs" / "gold_config.yml"

In [2]:
processor = GoldProcessor(config_path)
try:
    report = processor.run()
    print(f"Gold publicado: {report['release_id']}")
    print(f"Version Silver: {report['source']['version']}")
    print(f"Manifiesto: {report['current_manifest']}")
    for name, entry in report["tables"].items():
        print(f"{name}: {entry['rows']:,} filas")
finally:
    processor.close()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /opt/spark/ivy/cache
The jars for the packages stored in: /opt/spark/ivy/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-feab458e-9808-46e7-b93f-b3202df9d1ee;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 244ms :: artifacts dl 10ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0

Gold publicado: 41f1e426bbbb4561a97cca311dbc1a09
Version Silver: 0
Manifiesto: /opt/spark/exports/gold/current.json
fact_sales: 17,996,609 filas
fact_orders: 4,500,000 filas
dim_date: 2,557 filas
dim_customer: 299,994 filas
dim_product: 600,000 filas
dim_order_status: 3 filas
dim_market_segment: 5 filas
agg_sales_monthly: 450 filas
agg_customer_monthly: 4,062,601 filas
